In [32]:
import torch
from torch import nn, optim
from torch.nn.functional import mse_loss
from tqdm import tqdm, trange

state_dim = 8
message_dim = 32
action_dim = 1
hidden_dim = 256

In [33]:
class MessageActor(nn.Module):
    def __init__(self, state_dim: int, message_dim: int, hidden_dim: int = 256) -> None:
        super().__init__()
        self._net = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.LeakyReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.LeakyReLU(),
            nn.Linear(hidden_dim, message_dim),
        )

    def forward(self, state: torch.Tensor) -> torch.Tensor:
        return self._net(state)


class FinalActor(nn.Module):
    def __init__(self, state_dim: int, message_dim: int, action_dim: int, hidden_dim: int = 256) -> None:
        super().__init__()
        self._net = nn.Sequential(
            nn.Linear(state_dim + message_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.LeakyReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.LeakyReLU(),
            nn.Linear(hidden_dim, action_dim),
        )

    def forward(self, state: torch.Tensor, message: torch.Tensor) -> torch.Tensor:
        return self._net(torch.cat([state, message], dim=-1))


class MessageCritic(nn.Module):
    def __init__(self, state_dim: int, message_dim: int, hidden_dim: int = 256) -> None:
        super().__init__()
        self._net = nn.Sequential(
            nn.Linear(state_dim + message_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.LeakyReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.LeakyReLU(),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, state: torch.Tensor, message: torch.Tensor) -> torch.Tensor:
        return self._net(torch.cat([state, message], dim=-1))


class FinalCritic(nn.Module):
    def __init__(self, state_dim: int, message_dim: int, action_dim: int, hidden_dim: int = 256) -> None:
        super().__init__()
        self._net = nn.Sequential(
            nn.Linear(state_dim + message_dim + action_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.LeakyReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.LeakyReLU(),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, state: torch.Tensor, message: torch.Tensor, action: torch.Tensor) -> torch.Tensor:
        return self._net(torch.cat([state, message, action], dim=-1))

In [34]:
message_actor, target_message_actor = (MessageActor(state_dim, message_dim, hidden_dim) for _ in range(2))
final_actor, target_final_actor = (FinalActor(state_dim, message_dim, action_dim, hidden_dim) for _ in range(2))
message_critic, target_message_critic = (MessageCritic(state_dim, message_dim, hidden_dim) for _ in range(2))
final_critic1, target_final_critic1 = (FinalCritic(state_dim, message_dim, action_dim, hidden_dim) for _ in range(2))
final_critic2, target_final_critic2 = (FinalCritic(state_dim, message_dim, action_dim, hidden_dim) for _ in range(2))

online_models = [message_actor, final_actor, message_critic, final_critic1, final_critic2]
target_models = [target_message_actor, target_final_actor, target_message_critic, target_final_critic1, target_final_critic2]

for online, target in zip(online_models, target_models):
    target.load_state_dict(online.state_dict())
    for param in target.parameters():
        param.requires_grad_(False)

In [35]:
def reward_fn(state: torch.Tensor, action: torch.Tensor) -> torch.Tensor:
    return 1 - mse_loss(state.norm(dim=-1, keepdim=True), action, reduction="none")

In [36]:
def polyak_update(target: nn.Module, policy: nn.Module, tau: float) -> None:
    for target_param, policy_param in zip(target.parameters(), policy.parameters()):
        target_param.data.copy_(tau * policy_param.data + (1 - tau) * target_param.data)

In [37]:
lr = 3e-4
message_actor_opt = optim.Adam(message_actor.parameters(), lr=lr)
final_actor_opt = optim.Adam(final_actor.parameters(), lr=lr)
message_critic_opt = optim.Adam(message_critic.parameters(), lr=lr)
final_critic_opt = optim.Adam(
    list(final_critic1.parameters()) + list(final_critic2.parameters()),
    lr=lr,
)

In [40]:
num_steps = 1000000
batch_size = 128
epsilon = 0.05
epsilon_decay = 0.9999
epsilon_min = 0.001
tau = 0.0001
for step in trange(num_steps):
    state = torch.randn(batch_size, state_dim, requires_grad=True)

    # Train message actor
    message = message_actor(state)
    message = message + torch.randn_like(message) * epsilon
    action = target_final_actor(state, message)
    reward1 = final_critic1(state, message, action)
    reward2 = final_critic2(state, message, action)
    reward = torch.minimum(reward1, reward2)
    message_actor_opt.zero_grad()
    message_actor_loss = -reward.mean()
    message_actor_loss.backward()
    message_actor_opt.step()
    polyak_update(target_message_actor, message_actor, tau)

    # Train final actor
    message = target_message_actor(state)
    message = message + torch.randn_like(message) * epsilon
    action = final_actor(state, message)
    reward1 = final_critic1(state, message, action)
    reward2 = final_critic2(state, message, action)
    reward = torch.minimum(reward1, reward2)
    final_actor_opt.zero_grad()
    final_actor_loss = -reward.mean()
    final_actor_loss.backward()
    final_actor_opt.step()
    polyak_update(target_final_actor, final_actor, tau)

    # Train final critic
    message = target_message_actor(state)
    message = message + torch.randn_like(message) * epsilon
    action = final_actor(state, message)
    pred_reward1 = final_critic1(state, message, action)
    pred_reward2 = final_critic2(state, message, action)
    target_reward = reward_fn(state, action)
    final_critic_opt.zero_grad()
    final_critic_loss = mse_loss(pred_reward1, target_reward) + mse_loss(pred_reward2, target_reward)
    final_critic_loss.backward()
    final_critic_opt.step()
    polyak_update(target_final_critic1, final_critic1, tau)
    polyak_update(target_final_critic2, final_critic2, tau)

    epsilon = max(epsilon * epsilon_decay, epsilon_min)

    if step % 100 == 0:
        print(f"Step {step}, eps: {epsilon:.8f}, mal: {message_actor_loss:.8f}, fal: {final_actor_loss:.8f}, fcl: {final_critic_loss:.8f}, trw: {target_reward.mean():.8f}")

  0%|          | 8/1000000 [00:00<8:03:56, 34.44it/s]

Step 0, eps: 0.04999500, mal: -0.95507860, fal: -0.91023892, fcl: 0.00363665, trw: 0.97201717


  0%|          | 108/1000000 [00:03<8:09:53, 34.02it/s]

Step 100, eps: 0.04949752, mal: -1.06234050, fal: -1.01488757, fcl: 0.00286345, trw: 0.99057293


  0%|          | 208/1000000 [00:06<7:50:36, 35.41it/s]

Step 200, eps: 0.04900498, mal: -0.98292863, fal: -0.94987404, fcl: 0.00311955, trw: 0.98834026


... (92 outputs left out)

  0%|          | 4906/1000000 [02:23<8:18:18, 33.28it/s]

Step 4900, eps: 0.03062751, mal: -1.03311563, fal: -1.00033689, fcl: 0.00014173, trw: 0.99554121


  0%|          | 4908/1000000 [02:23<8:04:45, 34.21it/s]


KeyboardInterrupt: 